# ⛏️ 第二课 · 工作量证明

## 本课目标

- 理解"挖矿"的完整数学定义
- 看懂比特币区块头里的 nBits 字段到底在说什么
- 理解"难度"的数字含义——为什么现在是 70 万亿
- 理解为什么你的电脑永远挖不到比特币
- 理解区块时间为什么有时 1 分钟有时 1 小时

> 这一课的数学比第一课稍多，但高中数学够用。

## 1. 什么是"工作量证明"？

你是一个矿工。网络给你一个任务：

> "请找一个 nonce（随机数），使得 **SHA-256(SHA-256(区块头 + nonce))** 的结果小于某个目标值。"

你只能从 nonce=0 开始一个一个试。每次尝试叫一次"哈希"。

- 试了 100 次 → 你干了 100 次"活"
- 试了 1,000,000 次 → 你干了 100 万次"活"

这个"活"无法造假，因为 SHA-256 的结果他人可以一秒钟验证。

**这就是工作量证明（Proof of Work）**：用可量化的计算工作来证明你的付出。

## 2. nBits：难度目标如何编码

比特币区块头里有一个 32 位的字段叫 `nBits`。它用一种**压缩浮点数**格式存储难度目标：

```
target = mantissa × 256^(exponent - 3)

其中：
  exponent = nBits 的第一个字节
  mantissa = nBits 的后三个字节
```

创世区块的 nBits = `0x1d00ffff`：
- exponent = 0x1d = 29
- mantissa = 0x00ffff = 65535
- target = 65535 × 256^26

这是一个**非常非常大**的数（大约 2^224），意味着当时几乎任何 nonce 都能成功——因为全网只有中本聪一个人在挖。

In [ ]:
def bits_to_target(bits: int) -> int:
    # 把 nBits 解码成难度目标值
    exponent = bits >> 24          # 取第一个字节
    mantissa = bits & 0x00ffffff   # 取后三个字节
    return mantissa * (256 ** (exponent - 3))

def target_to_bits(target: int) -> int:
    # 把目标值编码成 nBits（逆运算）
    tb = target.to_bytes(32, "big")
    for i, b in enumerate(tb):
        if b != 0:
            exponent = 32 - i
            break
    else:
        return 0
    mantissa = target // (256 ** (exponent - 3))
    if mantissa >= 0x800000:  # 如果尾数太大，右移
        mantissa >>= 8
        exponent += 1
    return (exponent << 24) | mantissa

# 分析创世区块的 nBits
genesis_bits = 0x1d00ffff
genesis_target = bits_to_target(genesis_bits)

print(f"创世 nBits: 0x{genesis_bits:08x}")
print(f"创世目标:   0x{genesis_target:064x}")
print(f"编码往返:   {target_to_bits(genesis_target) == genesis_bits}")

# 当前典型的 nBits（2024 年）
current_bits = 0x17035e1a
current_target = bits_to_target(current_bits)
print(f"\n当前 nBits: 0x{current_bits:08x}")
print(f"当前目标:   0x{current_target:064x}")
print(f"目标缩小了:  {genesis_target / current_target:,.0f} 倍")

## 3. 难度 (Difficulty)

难度是一个**相对概念**：以创世区块的目标为基准，看现在难了多少倍。

```
difficulty = genesis_target / current_target
```

当前难度约 **70 万亿**。含义：现在挖一个块的难度是 2009 年的 70 万亿倍。

> 注意："难度"越高 → 目标值越小 → 越难找到有效哈希。难度和目标值是倒数关系。

In [ ]:
def target_to_difficulty(target: int) -> float:
    # 目标值 → 难度
    return bits_to_target(0x1d00ffff) / target

def difficulty_to_target(diff: float) -> int:
    # 难度 → 目标值
    return int(bits_to_target(0x1d00ffff) / diff)

# 不同时代难度的目标值变化
for label, diff in [
    ("2009 创世", 1),
    ("2010 GPU 时代", 10_000),
    ("2013 ASIC 降临", 500_000_000),
    ("2017 牛市", 1_000_000_000_000),
    ("2024 当前 (~70T)", 70_000_000_000_000),
]:
    t = difficulty_to_target(diff)
    leading_zeros = len(hex(t)) - 2  # 十六进制位数
    print(f"  {label:20s}: 难度 {diff:>20,}, 目标值占 {leading_zeros}/64 位十六进制")

## 4. 预期需要多少次哈希？

每次哈希的结果在 0 到 2^256-1 之间**均匀分布**（雪崩效应保证的）。

所以：
```
一次命中的概率 = target / 2^256
预期需要次数  = 1 / 概率 = 2^256 / target
```

这就是挖矿工作量的数学公式。难度越高 → target 越小 → 需要更多哈希。

In [ ]:
def expected_hashes(target: int) -> float:
    # 预期需要多少次哈希才能挖到一个块
    return (2 ** 256) / target

def expected_time(target: int, hashrate: float) -> float:
    # 给定算力，预期多久能挖到
    return expected_hashes(target) / hashrate

gen_target = bits_to_target(0x1d00ffff)
cur_target = difficulty_to_target(70_000_000_000_000)

def format_time(seconds):
    if seconds < 60:     return f"{seconds:.1f} 秒"
    if seconds < 3600:   return f"{seconds/60:.0f} 分钟"
    if seconds < 86400:  return f"{seconds/3600:.1f} 小时"
    if seconds < 86400*365: return f"{seconds/86400:.0f} 天"
    return f"{seconds/86400/365:,.0f} 年"

print(f"{'矿工':<25} | {'2009年出块':>12} | {'2024年出块':>14}")
print("-" * 60)
for name, hr in [
    ("你的 MacBook (CPU)", 10_000),
    ("高端 MacBook (GPU)", 3_000_000_000),
    ("1台 Antminer S21", 200_000_000_000_000),
    ("全网总算力", 600_000_000_000_000_000_000),
]:
    t_gen = expected_time(gen_target, hr)
    t_cur = expected_time(cur_target, hr)
    print(f"{name:<25} | {format_time(t_gen):>12} | {format_time(t_cur):>14}")

print()
print("💀 你的 MacBook 在 2024 年挖一个块期望时间 ≈ 宇宙年龄的 1000 倍")
print("💀 全网总算力也要约 10 分钟——这正是比特币协议的目标")

## 5. Nonce 不够用怎么办？

区块头里的 nonce 字段只有 **32 位**（4 字节），即 0 ~ 4,294,967,295。

一台 Antminer S21 算力约 200 TH/s = **每秒 2×10^14 次哈希**。

nonce 空间 ÷ 算力 = 4.3×10^9 ÷ 2×10^14 ≈ **0.00002 秒**就耗尽了。

那矿工怎么办？答案：**不只改 nonce**。

矿工还可以改：
- **coinbase 交易的 extraNonce 字段**（100 字节，无限空间）
- **时间戳**（每秒钟都可以加 1）

这些额外可改的字段合起来叫 **extranonce**，搜索空间远远超过 32 位的 nonce。

这也是矿池 Stratum 协议的核心——矿池分配不同的 extranonce 范围给不同矿工。

In [ ]:
NONCE_MAX = 2**32 - 1
asic_hashrate = 200 * 10**12  # 200 TH/s

exhaust_time = NONCE_MAX / asic_hashrate
print(f"Nonce 空间: 0 ~ {NONCE_MAX:,}")
print(f"ASIC 遍历完: {exhaust_time*1_000_000:.0f} 微秒")
print()
print("所以矿工必须改 coinbase 的 extraNonce + 时间戳来扩展空间")

## 6. 区块时间是随机的——这很正常

因为每次哈希是独立的、命中率极低，所以：

- **出块间隔服从指数分布**
- **每天出块数服从泊松分布**

这意味着：
- 有时候 1 分钟能出 3 个块
- 有时候 1 小时也不出一个块

**都是正常现象**，不是网络出问题了。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_blocks = 10000
target_time = 600  # 目标出块时间：10分钟 = 600秒
times = np.random.exponential(target_time, n_blocks)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：区块时间分布
axes[0].hist(times / 60, bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[0].axvline(x=10, color="red", linestyle="--", linewidth=2, label="目标=10分钟")
axes[0].set_xlabel("出块时间 (分钟)")
axes[0].set_ylabel("出现次数")
axes[0].set_title("区块时间分布（指数分布）")
axes[0].legend()
axes[0].set_xlim(0, 60)

# 右图：每日出块数的波动
# 每144个块大约对应1天
daily_blocks = []
for i in range(0, n_blocks - 144, 144):
    window_total = sum(times[i:i+144])
    daily_blocks.append(window_total / 600)  # 等效区块数
axes[1].hist(daily_blocks, bins=30, edgecolor="black", alpha=0.7)
axes[1].axvline(x=144, color="red", linestyle="--", linewidth=2, label="期望=144块/天")
axes[1].set_xlabel("每日区块数")
axes[1].set_ylabel("天数")
axes[1].set_title("每日区块数分布（泊松分布）")
axes[1].legend()

plt.tight_layout()
plt.savefig("../data/poisson_mining.png", dpi=100, bbox_inches="tight")
plt.close("all")
print("✅ 图表已保存到 data/poisson_mining.png")

## 本课小结

| 概念 | 一句话记住 |
|------|-----------|
| nBits | 32 位压缩浮点数，编码"目标值" |
| 难度 | = 1 号区块的目标 / 当前目标，目前 70 万亿 |
| 预期哈希 | = 2^256 / target，难度越高越难 |
| Nonce 溢出 | 32 位不够用，靠 extraNonce + 时间戳扩展 |
| 出块波动 | 泊松过程 → 1min 到 1h 都正常 |

**下一课**：打开一个真正的比特币区块，看看里面到底装了哪些东西。